[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/01-prerequisites/ml-prereq-calculus.ipynb)

# Calculus for ML

*AIBits Academy · Machine Learning End To End · Prerequisites*

Derivatives and gradients — the mathematical engine behind gradient descent, cost functions, and every "how do I improve this model" question.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## Why Calculus?

Linear algebra (previous page) gives data its shape — vectors, matrices, directions. But nothing about a matrix tells you which direction to *move* a set of weights to make a model better. That's calculus's job: it studies rates of change, and a cost function's rate of change with respect to each weight is exactly the signal every learning algorithm in this course follows downhill.

## The Derivative — Instantaneous Rate of Change

The derivative f'(x) answers: "if I nudge x by a tiny amount, how much does f(x) change?" Formally, it's the slope of the tangent line at a point — the limit of the slope between two points as they get infinitely close together.

$$f'(x) = \lim_{h\to 0}\frac{f(x+h)-f(x)}{h}$$

In [ ]:
import sympy as sp

x = sp.Symbol('x')
f = x**2 - 4*x + 7
fprime = sp.diff(f, x)

print(fprime)          # 2*x - 4
print(fprime.subs(x, 3))  # 2  -- the slope of f at x=3

### The Chain Rule

Most real cost functions are functions-of-functions: MSE is "square, then average" applied to "subtract" applied to a linear model's own output. The **chain rule** is what lets us differentiate through each layer of composition separately, then multiply the pieces together:

$$\frac{d}{dx}\big[g(h(x))\big] = g'(h(x))\cdot h'(x)$$

> **📊 Where this shows up**
>
> Every gradient derivation in this course — the Linear Regression page's ∇J(θ) = (1/m)Xᵀ(Xθ−y), the Logistic Regression sigmoid derivative, and Boosting's residual-fitting — is one chain-rule application after another. The "Deeper Insights" chapter on Boosting mentions the chain rule by name; this is the first-principles version of exactly that step.

## Interactive: The Tangent Line as a Sliding Window

Drag the point along the curve f(x) = x² − 4x + 7. The tangent line's slope *is* f'(x) = 2x − 4 evaluated at that x — watch it go negative (downhill), flatten to exactly zero at the minimum (x=2), then turn positive (uphill).

## Critical Points & Optimisation

A **critical point** is where f'(x) = 0 — a candidate minimum, maximum, or saddle. Setting the derivative to zero and solving is literally "find where the slope is flat":

In [ ]:
crit = sp.solve(fprime, x)
print(crit)              # [2]
print(f.subs(x, crit[0]))  # 3 -- the minimum value of f

### Newton-Raphson — Finding Roots Iteratively

Not every equation can be solved in closed form. Newton-Raphson finds where a function crosses zero by repeatedly following the tangent line down to the axis:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

In [ ]:
# Solve x^3 - x - 2 = 0, starting from x0=1
xn = 1.0
for i in range(6):
    fx  = xn**3 - xn - 2
    fpx = 3*xn**2 - 1
    xn  = xn - fx/fpx

print(xn)
# path: 1.0 -> 2.0 -> 1.6364 -> 1.5304 -> 1.5214 -> 1.52138 -> 1.52138 (converged)

Six iterations from a rough starting guess of x=1 land on the true root (≈1.52138) to 5 decimal places — this is the same convergence pattern used inside many optimisers, and directly analogous to how the Logistic Regression page's solver refines its coefficients.

### Interactive: Watching Newton-Raphson Walk Down to the Root

Step through the same six iterations on f(x) = x³ − x − 2. At each step, the tangent line at the current point is followed down to where it crosses the x-axis — that crossing point becomes the next guess, which is then carried straight up to the curve to start the next tangent.

## Convexity & the Cost Function

A function is **convex** if its second derivative f''(x) is always ≥ 0 — the graph never curves the "wrong way," so it has at most one minimum and no misleading local dips. For f(x) = x² − 4x + 7, f''(x) = 2 everywhere — constant and positive, so this parabola is convex at every point, meaning gradient descent starting anywhere is guaranteed to reach the true minimum.

> **📊 Prerequisite refresher**
>
> This is exactly the guarantee referenced on the Linear Regression page: "because J(θ) is a convex quadratic bowl (no local minima), gradient descent with a suitable learning rate is guaranteed to converge to the global optimum." Now you can verify that claim directly by taking the second derivative rather than taking it on faith.

## Gradient Descent — Repeatedly Stepping Downhill

Gradient descent is the critical-point idea turned into an iterative algorithm for functions too complex to solve for f'(x)=0 directly: take a small step in the direction opposite the gradient, repeat.

$$x_{k+1} = x_k - \alpha \cdot f'(x_k)$$

In [ ]:
xk, lr = 10.0, 0.1
path = [xk]
for i in range(15):
    grad = 2*xk - 4       # f'(x) = 2x - 4
    xk = xk - lr*grad
    path.append(round(xk, 4))

print(path)
# [10.0, 8.4, 7.12, 6.096, 5.2768, 4.6214, 4.0972, 3.6777,
#  3.3422, 3.0737, 2.859, 2.6872, 2.5498, 2.4398, 2.3518, 2.2815]
# -- steadily approaching the true minimum at x=2

## Interactive: Gradient Descent Racing Toward the Minimum

Watch the same 15-step descent from x₀=10 animate on the curve — each step's length is proportional to how steep the slope is at that point, which is exactly why steps shrink automatically as the ball nears the flat bottom.

## Multivariable Calculus — Partial Derivatives, the Gradient Vector, and the Hessian

Real cost functions depend on many parameters at once (every weight in a model). A **partial derivative** ∂J/∂θⱼ measures the slope in just one parameter's direction, holding all others fixed. Stack all the partials into a vector and you get the **gradient** ∇J — the direction of steepest ascent, which is exactly why gradient descent moves opposite to it.

In [ ]:
# 3-point toy dataset, fitting y = t0 + t1*x
X = [1, 2, 3]; Y = [3, 5, 7]
t0, t1 = sp.symbols('t0 t1')
J = sum((t0 + t1*xi - yi)**2 for xi, yi in zip(X, Y)) / (2*len(X))

dJ_dt0 = sp.diff(J, t0)   # t0 + 2*t1 - 5
dJ_dt1 = sp.diff(J, t1)   # 2*t0 + 14*t1/3 - 34/3

optimum = sp.solve([dJ_dt0, dJ_dt1], [t0, t1])
print(optimum)   # {t0: 1, t1: 2}  -- recovers y = 1 + 2x exactly

Solving where *both* partial derivatives equal zero simultaneously recovers θ₀=1, θ₁=2 — and (1,2,3)→(3,5,7) is exactly y=1+2x. This is the multivariable version of the single-variable critical-point idea above, generalised to a full parameter vector.

### The Hessian — Curvature in Every Direction at Once

The **Hessian** matrix collects all second partial derivatives — it's the multivariable generalisation of f''(x), and tells an optimiser about curvature (and therefore convexity) in every parameter direction simultaneously.

In [ ]:
H = sp.hessian(J, (t0, t1))
print(H)
# Matrix([[1, 2], [2, 14/3]])

> **📊 Prerequisite refresher**
>
> This Hessian is a symmetric matrix — exactly the matrix type introduced on the **Linear Algebra for ML** page. Newton's method for multivariable optimisation uses the Hessian's *inverse* to take smarter, curvature-aware steps instead of a fixed learning rate — the same inverse operation covered there.

## The Integral — Accumulating Area Under a Curve

Where the derivative measures instantaneous rate of change, the **integral** does the reverse: it accumulates a quantity across a range. The **Riemann sum** approximates this by slicing the region into thin rectangles and adding up their areas — and gets more accurate as the rectangles get thinner.

In [ ]:
# Demand curve: q(p) = 100 - 2p (units sold at price p), area = total "demand-price" mass, p=0 to 20
q = 100 - 2*x
exact = sp.integrate(q, (x, 0, 20))
print(exact)  # 1600  -- Fundamental Theorem of Calculus, exact answer

# Left-endpoint Riemann sum approximation
for n in [4, 100]:
    width = 20/n
    approx = sum(float(q.subs(x, i*width))*width for i in range(n))
    print(n, approx)
# 4   -> 1700.0  (coarse, 6.25% off)
# 100 -> 1604.0  (much closer to the exact 1600)

This is the **Fundamental Theorem of Calculus** in action: as the number of rectangles n→∞, the Riemann sum converges exactly to the integral. In ML, this same accumulation logic underlies computing the area under the ROC curve (AUC) on the Model Evaluation page, and probability density integration on the Inferential Statistics page.

### Log Scaling — A Calculus-Motivated Data Transform

The derivative of log(x) is 1/x — it shrinks fastest at small x and flattens at large x. That's precisely why taking log(y) compresses right-skewed data (revenue, prices, population) into something closer to linear/normal, which is the fix suggested for heteroscedastic residuals on the Linear Regression page's Q&A.

In [ ]:
import numpy as np

vals = np.array([100, 1000, 10000, 100000])
print(np.log10(vals))
# [2. 3. 4. 5.]  -- equal multiplicative jumps (10x each time) become equal additive steps

## A Note on Where This Stops (For Now)

> **⚠ Neural-network-specific calculus is out of scope here**
>
> Gradient descent, the chain rule, and the Hessian are the same mathematical objects used inside backpropagation, activation-function derivatives, and optimisers like Adam/RMSProp — but those are Neural Network-specific applications, and Deep Learning/Neural Networks are explicitly out of scope for this course. This page stops at the generic optimisation machinery (used directly by Linear Regression, Logistic Regression, and Boosting elsewhere in this course); backpropagation itself is a forward-reference to a future Deep Learning course, not built out here.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Find the critical points

With SymPy, differentiate `f = x**3 - 6*x**2 + 9*x`, solve `f' = 0`, and store the sorted critical points in `crit`.

In [ ]:
import sympy as sp
x = sp.Symbol("x")
f = x**3 - 6*x**2 + 9*x
crit = None   # TODO


In [ ]:
try:
    check("critical points are 1 and 3", crit == [1, 3])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import sympy as sp
x = sp.Symbol("x")
f = x**3 - 6*x**2 + 9*x
crit = sorted(sp.solve(sp.diff(f, x), x))

```

</details>

### Exercise 2 · Medium · Write gradient descent

Write `descend(grad, x0, lr, steps)` that repeatedly applies `x = x - lr * grad(x)` and returns the final `x`. It should find the minimum of `f(x) = (x - 3)**2` (gradient `2*(x - 3)`).

In [ ]:
def descend(grad, x0, lr, steps):
    pass   # TODO


In [ ]:
try:
    x_min = descend(lambda x: 2 * (x - 3), x0=10.0, lr=0.1, steps=200)
    check("converges to 3", abs(x_min - 3) < 1e-6)
    check("too-large learning rate diverges", abs(descend(lambda x: 2 * (x - 3), 10.0, 1.1, 50) - 3) > 100)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def descend(grad, x0, lr, steps):
    x = x0
    for _ in range(steps):
        x = x - lr * grad(x)
    return x

```

With lr = 1.1 each step overshoots by more than it corrects — the classic learning-rate blow-up.

</details>

### Exercise 3 · Stretch · Total demand by integration

Demand is `q(p) = 80 - 4*p` units at price `p`. Use `sp.integrate` to store in `area` the exact integral of `q` from p = 0 to p = 10, and in `riemann` the left-endpoint Riemann sum with 50 slices. Then set `error_pct` to how far the sum is from exact, as a percentage.

In [ ]:
import sympy as sp
p = sp.Symbol("p")
q = 80 - 4 * p
area = riemann = error_pct = None   # TODO


In [ ]:
try:
    check("exact area is 600", area == 600)
    check("riemann is close", 595 < riemann < 610)
    check("error % computed", abs(error_pct - abs(riemann - 600) / 600 * 100) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import sympy as sp
p = sp.Symbol("p")
q = 80 - 4 * p
area = sp.integrate(q, (p, 0, 10))
w = 10 / 50
riemann = sum(float(q.subs(p, i * w)) * w for i in range(50))
error_pct = abs(riemann - 600) / 600 * 100

```

</details>

---
*Back to the course: **Machine Learning End To End → Calculus for ML**.*